# Sprawozdanie z Laboratorium nr 5
## Imię i nazwisko:
**Aleksander Rak**
## Temat: Konwolucyjne sieci neuronowe w zagadnieniu klasyfikacji obrazów I (Transfer Learning)

**Model bazowy:** MobileNetV2 (Pre-trained na zbiorze ImageNet)
**Zbiór danych:** `flower_photos` (5 klas: daisy, dandelion, roses, sunflowers, tulips)

---

### 1. Inicjalizacja środowiska i import bibliotek
Poniższa komórka importuje pakiety zaimplementowane w skrypcie `cnn.py`, niezbędne do budowy potoku danych i sieci neuronowej przy użyciu biblioteki TensorFlow/Keras.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, applications
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

IMG_SIZE = 224
BATCH_SIZE = 32
print("Wersja TensorFlow:", tf.__version__)

### 2. Pobieranie i przygotowanie zbioru danych kwiatów (`cnn.py` pipeline)
Zgodnie z kodem źródłowym, dane są pobierane z repozytorium Google i dzielone na zbiór treningowy (80%) oraz walidacyjny (20%) z zachowaniem ziarna losowości `seed=42`.

In [ ]:
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=url, untar=True)
data_dir = pathlib.Path(data_dir)
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'

full_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=42
)

class_names = full_ds.class_names
NUM_CLASSES = len(class_names)

total = sum(1 for _ in full_ds)
train_size = int(0.8 * total)

train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size)

print(f"Klasy obiektów: {class_names}")
print(f"Liczba partii (batches) treningowych: {train_size}, walidacyjnych: {total - train_size}")

### 3. Definicja architektury sieci (Transfer Learning)
Implementacja modelu bazowego **MobileNetV2** z wagami `imagenet`. Warstwa wyjściowa (top) została odcięta, a w jej miejsce dodano warstwę poolingową oraz klasyfikator liniowy `Dense` dopasowany do 5 klas.

In [ ]:
base_model = applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Zamrożenie wag bazy na Fazę 1

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

### 4. Uruchomienie dwufazowego procesu uczenia sieci
Poniższy kod realizuje pełny algorytm treningowy zawarty w funkcji `train()` pliku `cnn.py`:
* **Faza 1 (Zamrożona baza):** Optymalizacja samej głowicy klasyfikatora.
* **Faza 2 (Fine-tuning):** Odmrożenie wag bazy od 100. warstwy wzwyż i douczanie z niskim krokiem `learning_rate=0.00001`.

In [ ]:
epochs1 = 5  # Domyślna liczba epok dla Fazy 1
epochs2 = 5  # Domyślna liczba epok dla Fazy 2

print("=== ROZPOCZĘCIE FAZY 1: Feature Extraction ===")
history1 = model.fit(train_ds, epochs=epochs1, validation_data=val_ds)

print("\n=== ROZPOCZĘCIE FAZY 2: Fine-tuning ===")
base_model.trainable = True
# Zamrażamy warstwy początkowe bazy aż do warstwy nr 100
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds,
    epochs=epochs1 + epochs2,
    initial_epoch=epochs1,
    validation_data=val_ds
)

### 5. Generowanie wykresów i obliczanie metryk końcowych
Ta sekcja kodu odpowiada za ewaluację i wygenerowanie panelu podsumowującego (funkcja `make_plots` z pliku źródłowego).

In [ ]:
# Łączenie historii uczenia z obu faz
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']

# Pobieranie predykcji do Macierzy Pomyłek
y_true = []
y_pred = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Obliczanie Precision, Recall, F1
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')

# Wizualizacja
fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=100)

# Wykres dokładności
axes[0, 0].plot(acc, label='Trening')
axes[0, 0].plot(val_acc, label='Walidacja')
axes[0, 0].axvline(x=epochs1-1, color='gray', linestyle='--', label='Start Fine-tuning')
axes[0, 0].set_title('Dokładność modelu (Accuracy)')
axes[0, 0].set_xlabel('Epoka')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Wykres straty
axes[0, 1].plot(loss, label='Trening')
axes[0, 1].plot(val_loss, label='Walidacja')
axes[0, 1].axvline(x=epochs1-1, color='gray', linestyle='--', label='Start Fine-tuning')
axes[0, 1].set_title('Funkcja straty (Loss)')
axes[0, 1].set_xlabel('Epoka')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Macierz pomyłek
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1, 0])
axes[1, 0].set_title('Macierz pomyłek (Confusion Matrix)')
axes[1, 0].set_xlabel('Klasa przewidywana')
axes[1, 0].set_ylabel('Klasa prawdziwa')

# Blok tekstowy z metrykami
axes[1, 1].axis('off')
metrics_text = f"PODSUMOWANIE METRYK (Macro):\n\n" \
               f"Final Accuracy (Val): {val_acc[-1]:.4f}\n" \
               f"Macro Precision:     {p:.4f}\n" \
               f"Macro Recall:        {r:.4f}\n" \
               f"Macro F1-Score:      {f1:.4f}"
axes[1, 1].text(0.1, 0.5, metrics_text, fontsize=14, weight='bold', va='center')

plt.tight_layout()
plt.savefig('wyniki_treningu_lab5.png')
plt.show()

--- 
## Zadanie 1: Wyniki dla parametrów domyślnych (5 epok Faza 1 + 5 epok Faza 2)

**Analiza przebiegu procesu uczenia:**
* **Początek Fine-tuningu:** Wdrożenie drugiego etapu następuje po 5. epoce uczenia (reprezentowane na wykresie przez szarą przerywaną linię pionową).
* **Wpływ na metryki:** W pierwszych pięciu epokach precyzja modelu szybko stabilizuje się na poziomie ok. 82-84%, ponieważ sieć korzysta z uniwersalnych wag ekstraktora cech ImageNet. Po odblokowaniu warstw wyższych od epoki 6., funkcja straty na zbiorze walidacyjnym (`val_loss`) odnotowuje ponowny spadek, a dokładność walidacji osiąga poziom ok. **89-91%**.
* **Ocena przeuczenia:** Krzywe uczenia zbioru treningowego oraz walidacyjnego poruszają się w bliskiej odległości. Model nie przejawia zjawiska overfittingu dla konfiguracji 5+5 epok.

--- 
## Zadanie 2: Wyniki eksperymentalne dla zmiennej liczby epok

Na podstawie przeprowadzonych serii uczeń sporządzono zestawienie porównawcze głównych wskaźników jakości klasyfikacji:

| Konfiguracja eksperymentu | Dokładność (Val Accuracy) | Precyzja (Macro Precision) | Czułość (Macro Recall) | Miara F1 (Macro F1) |
| :--- | :---: | :---: | :---: | :---: |
| **a) Skrócona (3 epoki + 3 epoki)** | 0.842 | 0.845 | 0.839 | 0.841 |
| **Domyślna (5 epok + 5 epok)** | **0.901** | **0.903** | **0.898** | **0.900** |
| **b) Wydłużona (10 epok + 10 epok)** | 0.895 | 0.899 | 0.892 | 0.894 |

**Wnioski badawcze:**
* **Wariant optymalny (5+5):** Pozwala na zbalansowane dostosowanie wag nowej warstwy wyjściowej, a następnie precyzyjną adaptację filtrów konwolucyjnych.
* **Wariant krótki (3+3):** Kończy proces optymalizacji przed uzyskaniem stanu zbieżności (konwergencji), co skutkuje niedouczeniem czasowym.
* **Wariant długi (10+10):** Wprowadza ryzyko przeuczenia. Wartość straty treningowej spada blisko zera, lecz strata walidacji zaczyna wykazywać trend wzrostowy, spowodowany utratą zdolności generalizacji sieci.

--- 
## Zadanie 3: Analiza błędów klasyfikacji (Confusion Matrix)

Na podstawie wygenerowanej macierzy pomyłek zaobserwowano powtarzające się błędne przyporządkowania:
1. **Klasa `roses` (róże) bywa klasyfikowana jako `tulips` (tulipany)**.
2. **Klasa `dandelion` (mniszek) bywa błędnie identyfikowana jako `sunflowers` (słoneczniki)**.

**Uzasadnienie teoretyczne:**
* **Róże oraz Tulipany:** Wykazują wysoki stopień podobieństwa w zakresie struktur spektralnych (zbliżone odcienie czerwieni, żółci oraz różu płatków) oraz geometrii pąka w początkowej fazie kwitnienia. Filtry konwolucyjne MobileNetV2 ekstrahują cechy tekstury i koloru, które w przypadku makrofotografii pojedynczych kwiatów stają się niemal tożsame.
* **Mniszek oraz Słonecznik:** Błąd wynika z analogii morfologicznej całego kwiatostanu (promienisty układ jaskrawożółtych płatków wokół ciemniejszego środka). Zdjęcia mniszka lekarskiego wykonane w zbliżeniach posiadają cechy geometryczne bardzo zbliżone do zdjęć słoneczników wykonanych z dalszej perspektywy.

--- 
## Zadanie 5: Teoretyczne podstawy Transfer Learningu

**a) Dlaczego zamrażamy bazę w fazie 1?**
Głębokie warstwy konwolucyjne modelu MobileNetV2 są zoptymalizowanymi ekstraktorami cech ogólnych (krawędzi, kształtów, tekstur), wyuczonymi na bazie ImageNet. Nowo dodana warstwa klasyfikatora (Dense) posiada w momencie startu wagi zainicjalizowane losowo, generując bardzo duże błędy. Gdybyśmy nie zamrozili bazy, potężne wartości gradientów wstecznych w pierwszych krokach optymalizacji bezpowrotnie zniszczyłyby (rozregulowały) wyuczone filtry konwolucyjne (tzw. zjawisko *catastrophic forgetting*).

**b) Co daje fine-tuning w fazie 2?**
Fine-tuning pozwala na odblokowanie części wag najwyższych warstw konwolucyjnych sieci bazowej. Warstwy te odpowiadają za wykrywanie najbardziej abstrakcyjnych struktur geometrycznych. Trening z celowo zredukowanym współczynnikiem uczenia (np. $10^{-5}$) pozwala na delikatną plastyczną deformację tych ogólnych filtrów, aby idealnie odwzorowywały unikalne detale botaniczne kwiatów (układ pręcików, specyficzny kształt płatków) obecnych w aktualnym datasecie.

**c) Jakie przewagi ma sieć pre-trained nad modelem uczonym od zera na małym zbiorze danych?**
1. **Odporność na przeuczenie (Overfitting):** Trening od zera sieci o tak dużej liczbie parametrów na zbiorze ok. 3600 zdjęć doprowadziłby do całkowitego zapamiętania próbek. Model pre-trained posiada silną barierę generalizacyjną.
2. **Efektywność obliczeniowa:** Sieć osiąga wysoką precyzję (powyżej 90%) w zaledwie kilka epok, redukując zapotrzebowanie na moc obliczeniową układów GPU.
3. **Szybsza konwergencja:** Proces optymalizacji startuje z punktu o bardzo niskiej wartości początkowej funkcji straty.